# INF 369 — Анализ и визуализация (Krisha, Алматы)

Полный разбор **`krisha_cleaned.csv`**: пять аналитических вопросов + расширенные графики по всем доступным полям.

**Графики** сохраняются в папку `charts/` (PNG + интерактивные HTML в Plotly).

Откройте ноутбук из корня проекта `data_final`, чтобы `Path.cwd()` указывал на CSV.


## Соответствие заданию (cleaning + analysis)

**Очистка данных (`preprocessing.py` → `krisha_cleaned.csv`):**
- **Пропуски:** удаление строк без цены/площади; медиана для `Rooms`; плейсхолдеры для текста; редкие категории сведены в «Прочее» (`House_Type_Group`, `Bathroom_Group`).
- **Дубликаты:** по `Listing_URL`, затем по (`Price_KZT`, `Area_m2`, `Address`).
- **Выбросы:** жёсткие правила по площади/цене + усечение 1–99 перцентиля по `Price_KZT`.
- **Подготовка к анализу:** `Market_Segment`, `Price_Tier`, `Build_Year_Bucket`, `Area_Band`, `District_Code`, `Is_NewBuild_Code`.

**Анализ (`analysis.py`):** шесть сформулированных вопросов (см. `ANALYTICAL_QUESTIONS`), описательные статистики, корреляции, групповые сравнения и блок **«простыми словами»** (`plain_language`).


In [ ]:
from pathlib import Path
import warnings

from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT = Path.cwd()
DATA_PATH = PROJECT / "krisha_cleaned.csv"
CHARTS = PROJECT / "charts"
CHARTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 160

PALETTE_SEG = {"Новостройка": "#2ecc71", "Вторичка": "#e74c3c"}

print("CSV:", DATA_PATH.resolve(), "exists:", DATA_PATH.exists())
print("Charts →", CHARTS.resolve())


In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["Is_NewBuild"] = df["Is_NewBuild"].astype(str).str.strip().str.lower().isin(
    ("true", "1", "yes", "1.0")
)
df["Segment"] = np.where(df["Is_NewBuild"], "Новостройка", "Вторичка")

# Производные признаки для графиков
def floor_level(row):
    c, t = row.get("Current_Floor"), row.get("Total_Floors")
    if pd.isna(c):
        return np.nan
    if c == 1:
        return "Первый"
    if not pd.isna(t) and c == t:
        return "Последний"
    return "Средние"


def building_type(tf):
    if pd.isna(tf):
        return np.nan
    if tf <= 5:
        return "≤5 эт. (хрущёвка)"
    if tf >= 12:
        return "≥12 эт. (высотка)"
    return "6–11 эт."


df["Floor_Level"] = df.apply(floor_level, axis=1)
df["Building_Type"] = df["Total_Floors"].apply(building_type)
df["Microdistrict"] = df["Address"].astype(str).str.split(",").str[0].str.strip()
df["Has_Developer"] = df["Developer"].notna() & (df["Developer"].astype(str).str.strip() != "")
df["Has_RC"] = df["Residential_Complex"].notna() & (df["Residential_Complex"].astype(str).str.strip() != "")
df["Rooms_label"] = df["Rooms"].apply(lambda r: "Студия" if r == 0 else f"{int(r)} комн.")

# Доля этажа (0–1), где известны оба
mask_ft = df["Current_Floor"].notna() & df["Total_Floors"].notna() & (df["Total_Floors"] > 0)
df["Floor_ratio"] = np.nan
df.loc[mask_ft, "Floor_ratio"] = (
    df.loc[mask_ft, "Current_Floor"].astype(float) / df.loc[mask_ft, "Total_Floors"].astype(float)
)

print(f"Строк: {len(df):,} | Районов: {df['District'].nunique()} | Сегменты:")
print(df["Segment"].value_counts())
print("\nПропуски по ключевым полям:")
for col in ["Build_Year", "House_Type", "Ceiling_Height", "Bathroom", "Developer", "Residential_Complex", "Current_Floor"]:
    if col in df.columns:
        print(f"  {col}: {df[col].isna().mean()*100:.1f}%")
df.head(3)


## Задача 3 — пять аналитических вопросов

Импортируем `analyze` из `analysis.py` и выводим те же таблицы, что и в консольном пайплайне.


In [ ]:
from analysis import analyze, ANALYTICAL_QUESTIONS

results = analyze(str(DATA_PATH))

print("=== Формулировки аналитических вопросов ===")
for k, v in results["questions"].items():
    print(f"  {k}: {v}")

print("\n=== Описательная статистика (числовые поля) ===")
display(results["describe_numeric"])

print("\n=== Корреляции (факторы и цена за м²) ===")
display(results["correlations"])

print("\n=== Q1 — районы ===")
display(results["q1"])
print("\n=== Q2 — комнаты vs ₸/м² ===")
print("Pearson:", results["q2"]["pearson_corr"], "| Spearman:", results["q2"]["spearman_corr"])
display(results["q2"]["by_room_count"])
print("\n=== Q3 — позиция этажа ===")
display(results["q3"])
print("\n=== Q4 — тип высотности дома ===")
display(results["q4"])
print("\n=== Q5 — микрорайоны ===")
display(results["q5"])
print("\n=== Q6 — год постройки vs ₸/м² (≥3 объявлений на год) ===")
display(results["q6"])

print("\n=== Выводы простыми словами (для нетехнической аудитории) ===")
for line in results["plain_language"]:
    print(" •", line)


## Визуализация — новые графики

Ниже — интерактивные и статические графики, использующие район, сегмент, комнатность, тип дома, санузел, застройщика, ЖК, год постройки, высоту потолков, этажность и площадь.


In [ ]:
def fmt_k_short(x, pos=None):
    xa = float(x)
    if abs(xa) >= 1e6:
        return f"{xa/1e6:.1f}M"
    if abs(xa) >= 1e3:
        return f"{xa/1e3:.0f}K"
    return f"{xa:.0f}"


# --- 1. Sunburst: район → сегмент → комнатность ---
room_order = ["Студия", "1 комн.", "2 комн.", "3 комн.", "4 комн.", "5 комн.", "6+ комн."]
df["Rooms_cat"] = pd.cut(
    df["Rooms"].clip(0, 6),
    bins=[-0.1, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 100],
    labels=["Студия", "1 комн.", "2 комн.", "3 комн.", "4 комн.", "5 комн.", "6+ комн."],
)
df_sb = df.dropna(subset=["Rooms_cat"])
agg_sb = (
    df_sb.groupby(["District", "Segment", "Rooms_cat"], observed=True)
    .size()
    .reset_index(name="count")
)
fig = px.sunburst(
    agg_sb,
    path=["District", "Segment", "Rooms_cat"],
    values="count",
    title="Структура рынка: район → новостройка/вторичка → комнатность",
    color="Segment",
    color_discrete_map=PALETTE_SEG,
)
fig.update_layout(margin=dict(t=50, l=0, r=0, b=0), height=700)
fig.write_html(CHARTS / "01_sunburst_district_segment_rooms.html")
fig.show()
print("Saved", CHARTS / "01_sunburst_district_segment_rooms.html")


In [ ]:
# --- 2. Тепловая карта: медиана ₸/м² (район × комнатность) ---
pivot = (
    df.groupby(["District", "Rooms"])["Price_per_m2"]
    .median()
    .unstack("Rooms")
    .reindex(columns=sorted(df["Rooms"].dropna().unique()))
)
plt.figure(figsize=(12, 6))
sns.heatmap(
    pivot / 1000,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Медиана ₸/м² (тыс.)"},
)
plt.title("Медианная цена за м² (тыс. ₸): район × число комнат")
plt.xlabel("Комнат")
plt.ylabel("Район")
plt.tight_layout()
plt.savefig(CHARTS / "02_heatmap_median_price_m2_district_rooms.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 3. «Гребенка» KDE: распределение ₸/м² по районам ---
districts = df["District"].value_counts().index.tolist()
n = len(districts)
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n), sharex=True)
if n == 1:
    axes = [axes]
for ax, dist in zip(axes, districts):
    sub = df.loc[df["District"] == dist, "Price_per_m2"].dropna() / 1000
    if len(sub) > 1:
        sns.kdeplot(sub, ax=ax, fill=True, alpha=0.7, color="#3498db")
    ax.set_ylabel(dist, rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
axes[-1].set_xlabel("Цена за м², тыс. ₸")
fig.suptitle("Плотность распределения цены за м² по районам", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(CHARTS / "03_ridge_kde_price_per_m2_by_district.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 4. Violin: тип дома × сегмент (топ-6 типов по числу объявлений) ---
ht = df.dropna(subset=["House_Type"])
top_ht = ht["House_Type"].value_counts().head(6).index
sub_ht = ht[ht["House_Type"].isin(top_ht)]
plt.figure(figsize=(11, 6))
sns.violinplot(
    data=sub_ht,
    x="House_Type",
    y="Price_per_m2",
    hue="Segment",
    split=True,
    inner="quart",
    palette=PALETTE_SEG,
    hue_order=["Новостройка", "Вторичка"],
    cut=0,
)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k_short))
plt.xticks(rotation=20, ha="right")
plt.title("Распределение ₸/м² по типу дома (топ-6) и сегменту")
plt.xlabel("")
plt.ylabel("₸/м²")
plt.legend(title="Сегмент", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(CHARTS / "04_violin_housetype_segment.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 5. Топ ЖК по медиане ₸/м² (минимум 5 листингов) ---
rc = df[df["Residential_Complex"].notna() & (df["Residential_Complex"].astype(str).str.strip() != "")]
agg_rc = (
    rc.groupby("Residential_Complex")
    .agg(median_m2=("Price_per_m2", "median"), n=("Listing_ID", "count"))
    .query("n >= 5")
    .sort_values("median_m2", ascending=False)
    .head(15)
)
plt.figure(figsize=(10, 7))
sns.barplot(x=agg_rc["median_m2"] / 1000, y=agg_rc.index, palette="magma")
plt.xlabel("Медиана ₸/м² (тыс.)")
plt.title("Топ-15 ЖК по медианной цене за м² (n ≥ 5)")
plt.tight_layout()
plt.savefig(CHARTS / "05_top_residential_complexes_median_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 6. Топ застройщиков: число листингов и медиана ₸/м² ---
dev = df[df["Has_Developer"]]
agg_d = (
    dev.groupby("Developer")
    .agg(n=("Listing_ID", "count"), med=("Price_per_m2", "median"))
    .query("n >= 3")
    .sort_values("n", ascending=False)
    .head(12)
)
fig, ax1 = plt.subplots(figsize=(10, 6))
x = np.arange(len(agg_d))
ax1.barh(x, agg_d["n"], color="#9b59b6", alpha=0.85, label="Листингов")
ax1.set_yticks(x)
ax1.set_yticklabels(agg_d.index)
ax1.set_xlabel("Число объявлений")
ax1.invert_yaxis()
ax2 = ax1.twiny()
ax2.scatter(agg_d["med"] / 1000, x, color="#e67e22", s=80, zorder=3, label="Медиана ₸/м² (тыс.)")
ax2.set_xlabel("Медиана ₸/м² (тыс.)")
ax1.set_title("Застройщики: объём (столбцы) и медиана ₸/м² (точки), n ≥ 3")
fig.tight_layout()
fig.savefig(CHARTS / "06_developers_count_and_median_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 7. Санузел: медиана ₸/м² и количество ---
bath = df.dropna(subset=["Bathroom"])
order_b = bath["Bathroom"].value_counts().index.tolist()
plt.figure(figsize=(10, 5))
sns.boxplot(data=bath, x="Bathroom", y="Price_per_m2", order=order_b, palette="Set2", showfliers=False)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k_short))
plt.xticks(rotation=25, ha="right")
plt.title("Цена за м² по типу санузла")
plt.xlabel("")
plt.ylabel("₸/м²")
plt.tight_layout()
plt.savefig(CHARTS / "07_boxplot_bathroom_price_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 8. Год постройки × сегмент ---
by = df.dropna(subset=["Build_Year"])
plt.figure(figsize=(10, 5))
for seg, color in PALETTE_SEG.items():
    sub = by.loc[by["Segment"] == seg, "Build_Year"]
    plt.hist(sub, bins=range(int(by["Build_Year"].min()), int(by["Build_Year"].max()) + 2), alpha=0.55, label=seg, color=color, density=True)
plt.xlabel("Год постройки")
plt.ylabel("Плотность")
plt.title("Распределение года постройки: новостройка vs вторичка")
plt.legend()
plt.tight_layout()
plt.savefig(CHARTS / "08_hist_build_year_by_segment.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 9. Высота потолков vs ₸/м² ---
ch = df.dropna(subset=["Ceiling_Height", "Price_per_m2"])
plt.figure(figsize=(9, 6))
sns.scatterplot(data=ch, x="Ceiling_Height", y="Price_per_m2", hue="Segment", alpha=0.45, palette=PALETTE_SEG, s=35)
sns.regplot(data=ch, x="Ceiling_Height", y="Price_per_m2", scatter=False, color="#34495e", line_kws={"linestyle": "--"})
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k_short))
plt.title("Высота потолков и цена за м² (общий тренд — пунктир)")
plt.xlabel("Потолки, м")
plt.ylabel("₸/м²")
plt.legend(title="Сегмент")
plt.tight_layout()
plt.savefig(CHARTS / "09_scatter_ceiling_vs_price_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 10. Позиция этажа (первый / средний / последний) ---
fl = df.dropna(subset=["Floor_Level"])
plt.figure(figsize=(8, 5))
order_fl = ["Первый", "Средние", "Последний"]
sns.boxplot(data=fl, x="Floor_Level", y="Price_KZT", order=order_fl, palette="pastel", showfliers=False)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k_short))
plt.title("Цена объекта (₸) по позиции этажа")
plt.xlabel("")
plt.ylabel("Цена, ₸")
plt.tight_layout()
plt.savefig(CHARTS / "10_boxplot_floor_level_total_price.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 11. Hexbin: площадь × цена ---
plt.figure(figsize=(9, 7))
hb = plt.hexbin(df["Area_m2"], df["Price_KZT"] / 1e6, gridsize=35, cmap="viridis", mincnt=1)
plt.colorbar(hb, label="Число объявлений в ячейке")
plt.xlabel("Площадь, м²")
plt.ylabel("Цена, млн ₸")
plt.title("Плотность объявлений: площадь vs цена")
plt.tight_layout()
plt.savefig(CHARTS / "11_hexbin_area_vs_price_mln.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 12. Корреляции числовых признаков (heatmap; без дендрограмм — устойчиво к NaN) ---
num_cols = ["Price_KZT", "Price_per_m2", "Area_m2", "Rooms", "Current_Floor", "Total_Floors", "Build_Year", "Ceiling_Height", "Floor_ratio"]
num_cols = [c for c in num_cols if c in df.columns]
sub = df[num_cols].dropna(how="any")
# убрать константные столбцы — иначе corr даёт NaN
sub = sub.loc[:, sub.std(numeric_only=True) > 1e-12]
if len(sub) >= 30 and sub.shape[1] >= 2:
    cm = sub.corr(numeric_only=True)
    cm = cm.replace([np.inf, -np.inf], np.nan).fillna(0)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, linewidths=0.5)
    plt.title("Корреляции числовых признаков (полные строки)")
    plt.tight_layout()
    plt.savefig(CHARTS / "12_correlation_heatmap_numeric.png", dpi=160, bbox_inches="tight")
    plt.close("all")
    print("Saved", CHARTS / "12_correlation_heatmap_numeric.png")
else:
    print("Недостаточно данных для heatmap:", len(sub), "строк,", sub.shape[1] if len(sub) else 0, "колонок")


In [ ]:
# --- 13. Treemap: доля листингов по району и сегменту ---
agg_tr = df.groupby(["District", "Segment"]).size().reset_index(name="count")
fig = px.treemap(
    agg_tr,
    path=["District", "Segment"],
    values="count",
    color="Segment",
    color_discrete_map=PALETTE_SEG,
    title="Доля объявлений: район → сегмент",
)
fig.update_layout(margin=dict(t=50, l=0, r=0, b=0), height=600)
fig.write_html(CHARTS / "13_treemap_district_segment.html")
fig.show()


In [ ]:
# --- 14. Parallel categories (сэмпл до 1500 для скорости) ---
sample = df.sample(min(1500, len(df)), random_state=42)
cats = sample.copy()
cats["Rooms_cat"] = cats["Rooms"].clip(0, 5).astype(int).astype(str) + " к."
cats["Building_Type"] = cats["Building_Type"].fillna("неизвестно").astype(str)
fig = px.parallel_categories(
    cats,
    dimensions=["District", "Segment", "Rooms_cat", "Building_Type"],
    color=sample["Price_per_m2"].values,
    color_continuous_scale="Turbo",
    title="Связки: район → сегмент → комнат → тип высотности (цвет = ₸/м²)",
)
fig.update_layout(height=700, margin=dict(t=60, b=20))
fig.write_html(CHARTS / "14_parallel_categories_price_m2.html")
fig.show()


In [ ]:
# --- 15. Стек: комнатность в разрезе района и сегмента (топ-6 районов по объёму) ---
top_d = df["District"].value_counts().head(6).index
sub = df[df["District"].isin(top_d)]
ct = pd.crosstab([sub["District"], sub["Segment"]], sub["Rooms_label"])
ct.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="tab20", edgecolor="white", linewidth=0.4)
plt.title("Топ-6 районов: структура по комнатности (стек), разбивка по сегменту на оси X")
plt.xlabel("Район + сегмент")
plt.ylabel("Число объявлений")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Комнатность", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(CHARTS / "15_stacked_rooms_district_segment.png", bbox_inches="tight")
plt.show()


## Итог

Все новые файлы лежат в **`charts/`** с префиксами `01_`…`15_`. HTML откройте в браузере для интерактива.

При обновлении данных перезапустите ноутбук целиком (**Run All**).
